# Data Collection and Processing Pipeline

This notebook builds the dataset for our software defect prediction project from scratch. The goal of the project is to predict whether AI-generated code will pass its test suite without actually running it. To train models for that, we need a large collection of (code, pass/fail label) pairs.

The BigCodeBench benchmark provides exactly what we need: 1,140 Python programming tasks, code generated by 100+ LLMs for each task, and pass/fail evaluation results. The raw data lives across three separate sources (a HuggingFace dataset, a GitHub release zip, and two more HuggingFace performance datasets), so the bulk of this notebook is about downloading those sources and stitching them together into a single flat CSV.

By the end, we produce `processed/samples.csv` with 123,416 rows covering 57 models.

## Setup

In [1]:
import json
import re
import zipfile
from pathlib import Path

import pandas as pd
import requests
from datasets import load_dataset, load_dataset_builder
from tqdm.notebook import tqdm

In [2]:
RAW = Path("raw")
PROCESSED = Path("processed")

TASKS_PATH = RAW / "bigcodebench_tasks.jsonl"
SAMPLES_ZIP = RAW / "sanitized_calibrated_samples.zip"
SAMPLES_DIR = RAW / "sanitized_calibrated_samples"
EVAL_DIR = RAW / "eval_results"

RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

## Step 1: Download the task descriptions

BigCodeBench is a benchmark of 1,140 Python programming tasks published by the BigCode team on HuggingFace (`bigcode/bigcodebench`). Each task includes:

- A **complete prompt** (docstring-style, with function signature, argument docs, and examples)
- An **instruct prompt** (a short natural language description of the same task)
- A canonical solution, unit tests, the entry point function name, and required libraries

We use split `v0.1.4`, which is the stable release that matches the evaluation results we download later. The tasks themselves are the same across versions, but the split name matters because the eval results were generated against this version.

We save the tasks as JSONL so we can load them line-by-line later without needing the `datasets` library at merge time.

In [3]:
if TASKS_PATH.exists():
    print(f"Tasks already downloaded at {TASKS_PATH}, skipping.")
else:
    print("Downloading BigCodeBench tasks from HuggingFace...")
    tasks_ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
    tasks_ds.to_json(str(TASKS_PATH))
    print(f"Saved {len(tasks_ds)} tasks to {TASKS_PATH}")

Tasks already downloaded at raw/bigcodebench_tasks.jsonl, skipping.


Let's take a quick look at what a task looks like.

In [4]:
with open(TASKS_PATH) as f:
    sample_task = json.loads(f.readline())

print(f"Task ID: {sample_task['task_id']}")
print(f"Entry point: {sample_task['entry_point']}")
print(f"Libraries: {sample_task['libs']}")
print(f"\nComplete prompt (first 300 chars):\n{sample_task['complete_prompt'][:300]}...")
print(f"\nInstruct prompt:\n{sample_task['instruct_prompt'][:300]}...")

Task ID: BigCodeBench/0
Entry point: task_func
Libraries: ['random', 'itertools']

Complete prompt (first 300 chars):
import itertools
from random import shuffle

def task_func(numbers=list(range(1, 3))):
    """
    Calculates the average of the sums of absolute differences between each pair of consecutive numbers 
    for all permutations of a given list. Each permutation is shuffled before calculating the differ...

Instruct prompt:
Calculates the average of the sums of absolute differences between each pair of consecutive numbers for all permutations of a given list. Each permutation is shuffled before calculating the differences. Args: - numbers (list): A list of numbers. Default is numbers from 1 to 10.
The function should o...


## Step 2: Download LLM-generated code samples

The BigCodeBench team ran over 100 LLMs on all 1,140 tasks and published the generated code as a zip file on their GitHub releases page (v0.2.5). The zip is about 158 MB and contains JSONL files organized by prompt format.

The zip has six subdirectories:
- `complete/` — 87 model files, code generated from the docstring-style prompt
- `instruct/` — 134 model files, code generated from the short natural language prompt
- `full/complete/` and `full/instruct/` — newer models (DeepSeek-R1, Qwen2.5, Llama-4, etc.)
- `hard/complete/` and `hard/instruct/` — a 148-task subset of harder problems

We only use `complete/` and `instruct/`. The `full/` directories contain newer models that don't have corresponding pass/fail evaluation results yet, so we'd end up with code samples but no labels. The `hard/` directories only cover 148 of the 1,140 tasks, which would leave most of our data without coverage.

Each JSONL file has one line per task, with fields `task_id` and `solution`.

In [5]:
if SAMPLES_DIR.exists() and any(SAMPLES_DIR.glob("**/*.jsonl")):
    print(f"Samples already extracted at {SAMPLES_DIR}, skipping.")
else:
    if not SAMPLES_ZIP.exists():
        url = "https://github.com/bigcode-project/bigcodebench/releases/download/v0.2.5/sanitized_calibrated_samples.zip"
        print(f"Downloading code samples from GitHub ({url})...")
        resp = requests.get(url, stream=True, timeout=300)
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))
        with open(SAMPLES_ZIP, "wb") as fh, tqdm(total=total, unit="B", unit_scale=True, desc="Download") as bar:
            for chunk in resp.iter_content(1 << 20):  # 1 MB chunks
                fh.write(chunk)
                bar.update(len(chunk))
        print(f"Download complete: {SAMPLES_ZIP}")

    print("Extracting zip...")
    with zipfile.ZipFile(SAMPLES_ZIP) as zf:
        zf.extractall(RAW)
    print(f"Extracted to {SAMPLES_DIR}")

Samples already extracted at raw/sanitized_calibrated_samples, skipping.


Let's see what we got.

In [6]:
for subdir in sorted(SAMPLES_DIR.iterdir()):
    if subdir.is_dir():
        n_files = len(list(subdir.glob("**/*.jsonl")))
        print(f"  {subdir.name}/: {n_files} model files")

  complete/: 87 model files
  full/: 250 model files
  hard/: 101 model files
  instruct/: 134 model files


In [7]:
# peek at one sample file to see the format
sample_file = next((SAMPLES_DIR / "complete").glob("*.jsonl"))
with open(sample_file) as f:
    sample_row = json.loads(f.readline())

print(f"File: {sample_file.name}")
print(f"Task ID: {sample_row['task_id']}")
print(f"Solution (first 200 chars):\n{sample_row['solution'][:200]}...")

File: deepseek-ai--deepseek-coder-33b-instruct--bigcodebench-complete--vllm-0-1-sanitized-calibrated.jsonl
Task ID: BigCodeBench/0
Solution (first 200 chars):
import itertools
from random import shuffle
def task_func(numbers=list(range(1, 3))):
    """
    Calculates the average of the sums of absolute differences between each pair of consecutive numbers 
 ...


## Step 3: Download pass/fail evaluation results

The BigCodeBench team also published per-model evaluation results on HuggingFace as two separate datasets:
- `bigcode/bigcodebench-complete-perf` — 85 model splits (one split per model)
- `bigcode/bigcodebench-instruct-perf` — 52 model splits

Each split contains rows with `task_id` and `status` (1 = passed all unit tests, 0 = failed). We download every split and save each one as a standalone JSON file mapping task IDs to pass/fail labels. This gives us 137 JSON files total.

We save them as individual files rather than loading everything into memory so we can inspect and debug specific models easily.

In [8]:
existing_evals = list(EVAL_DIR.glob("*.json"))

if len(existing_evals) >= 137:
    print(f"Eval results already downloaded ({len(existing_evals)} files), skipping.")
else:
    print("Downloading evaluation results from HuggingFace...")

    for ds_name, split_tag in [
        ("bigcode/bigcodebench-complete-perf", "complete"),
        ("bigcode/bigcodebench-instruct-perf", "instruct"),
    ]:
        builder = load_dataset_builder(ds_name)
        model_splits = list(builder.info.splits.keys())
        print(f"  {split_tag}: {len(model_splits)} models")

        for model_split in tqdm(model_splits, desc=split_tag):
            out_path = EVAL_DIR / f"{model_split}--{split_tag}_eval_results.json"
            if out_path.exists():
                continue
            ds = load_dataset(ds_name, split=model_split)
            labels = {row["task_id"]: int(row["status"]) for row in ds}
            with open(out_path, "w") as fh:
                json.dump(labels, fh)

    print(f"Downloaded {len(list(EVAL_DIR.glob('*.json')))} eval result files.")

Eval results already downloaded (137 files), skipping.


In [9]:
# quick look at one eval file
eval_files = sorted(EVAL_DIR.glob("*.json"))
print(f"Total eval files: {len(eval_files)}")
print(f"\nFirst few files:")
for f in eval_files[:5]:
    print(f"  {f.name}")

with open(eval_files[0]) as fh:
    sample_labels = json.load(fh)
print(f"\nSample eval file has {len(sample_labels)} task labels.")
print(f"First 3 entries: {dict(list(sample_labels.items())[:3])}")

Total eval files: 137

First few files:
  AutoCoder--complete_eval_results.json
  AutoCoder--instruct_eval_results.json
  AutoCoder_QW_7B--complete_eval_results.json
  AutoCoder_QW_7B--instruct_eval_results.json
  AutoCoder_S_6.7B--complete_eval_results.json

Sample eval file has 1140 task labels.
First 3 entries: {'BigCodeBench/0': 1, 'BigCodeBench/1': 1, 'BigCodeBench/2': 1}


## Step 4: Merge samples with labels

This is where it gets tricky. The code sample files and the evaluation result files use different naming conventions for the same models. For example, the zip file names a model `codellama--CodeLlama-7b-Instruct-hf` while the eval dataset calls it `CodeLlama_7B_Instruct`.

We handle this with a two-stage matching approach:

1. **Fuzzy normalization**: strip the organization prefix (everything before `--`), remove the `-hf` suffix, remove separators and version markers, and lowercase. This catches most cases.

2. **Manual mapping**: for the cases where normalization still doesn't produce a match (API-style names like `gemini-1.5-flash`, version suffixes like `-v0.1`, org-prefix quirks), we maintain an explicit dictionary.

Models that can't be matched at all are dropped. This mostly affects base (non-instruction-tuned) models that appear in the eval datasets but not in the zip, since the zip only contains instruct/chat variants.

### 4a. Define the manual mapping

These are the model names where fuzzy normalization fails. Each key is the model identifier as it appears in the zip filename, and each value is the corresponding name from the evaluation dataset. We identified these by running the normalization, collecting the unmatched models, and looking up the correct correspondences by hand.

In [10]:
MANUAL_MAP = {
    "codellama--CodeLlama-7b-Instruct-hf":        "CodeLlama_7B_Instruct",
    "codellama--CodeLlama-13b-Instruct-hf":       "CodeLlama_13B_Instruct",
    "codellama--CodeLlama-34b-Instruct-hf":       "CodeLlama_34B_Instruct",
    "codellama--CodeLlama-70b-Instruct-hf":       "CodeLlama_70B_Instruct",
    "mistralai--Mixtral-8x22B-Instruct-v0.1":     "Mixtral_8x22B_Instruct",
    "mistralai--Mistral-7B-Instruct-v0.3":        "Mistral_7B_Instruct_v0.3",
    "gemini-1.5-flash":                           "Gemini_1.5_Flash_API_0514",
    "gemini-1.5-pro":                             "Gemini_1.5_Pro_API_0514",
    "google--codegemma-7b-it":                    "CodeGemma_7B_Instruct",
    "google--gemma-2-9b-it":                      "Gemma_2_9B_Instruct",
    "meta-llama--Meta-Llama-3-8B-Instruct":       "Llama_3_8B_Instruct",
    "meta-llama--Meta-Llama-3-70B-Instruct":      "Llama_3_70B_Instruct",
    "meta-llama--Meta-Llama-3.1-8B-Instruct":     "Llama_3_8B_Instruct",
    "meta-llama--Meta-Llama-3.1-70B-Instruct":    "Llama_3_70B_Instruct",
    "bigcode--starcoder2-15b-instruct-v0.1":      "StarCoder2_15B_Instruct_v0.1",
    "CohereForAI--c4ai-command-r-plus":           "Command_R_plus",
    "ibm-granite--granite-3b-code-instruct":      "Granite_Code_3B_Instruct",
    "ibm-granite--granite-8b-code-instruct":      "Granite_Code_8B_Instruct",
    "ibm-granite--granite-20b-code-instruct":     "Granite_Code_20B_Instruct",
    "ibm-granite--granite-34b-code-instruct":     "Granite_Code_34B_Instruct",
}

### 4b. Define the normalization function

This function takes a model identifier from either source and reduces it to a minimal canonical form for comparison. The steps are:

1. Strip the organization prefix (e.g., `codellama--CodeLlama-7b` becomes `CodeLlama-7b`)
2. Remove the `-hf` suffix that HuggingFace appends to some model names
3. Remove all separators (hyphens, underscores, dots, spaces) and version markers
4. Strip trailing date stamps (8-digit suffixes like `20240229`)
5. Lowercase everything

After this, names like `CodeLlama-7b-Instruct-hf` and `CodeLlama_7B_Instruct` both become `codellama7binstruct`.

In [11]:
def normalize_model_name(name):
    """Reduce a model identifier to a minimal form for fuzzy matching."""
    if "--" in name:
        name = name.split("--", 1)[1]
    name = re.sub(r"-hf$", "", name)
    name = re.sub(r"[-_v. ]", "", name)
    name = re.sub(r"\d{8}$", "", name)
    return name.lower()

In [12]:
# verify it works on a few examples
test_cases = [
    ("codellama--CodeLlama-7b-Instruct-hf", "codellama7binstruct"),
    ("deepseek-ai--deepseek-coder-33b-instruct", "deepseekcoder33binstruct"),
    ("Claude_3_Opus_20240229", "claude3opus"),
]
for raw, expected in test_cases:
    result = normalize_model_name(raw)
    status = "ok" if result == expected else f"MISMATCH (got {result})"
    print(f"  {raw:50s} -> {result:30s} [{status}]")

  codellama--CodeLlama-7b-Instruct-hf                -> codellama7binstruct            [ok]
  deepseek-ai--deepseek-coder-33b-instruct           -> deepseekcoder33binstruct       [ok]
  Claude_3_Opus_20240229                             -> claude3opus                    [ok]


### 4c. Load eval labels into a lookup structure

We load all evaluation JSON files for a given split (complete or instruct) into a dictionary. The keys are model names as they appear in the eval filenames, and the values are dicts mapping task IDs to pass/fail labels.

In [13]:
def load_eval_lookup(split):
    """Load all eval result files for a split into {model_name: {task_id: label}}."""
    lookup = {}
    for f in EVAL_DIR.glob(f"*--{split}_eval_results.json"):
        model = f.stem.replace(f"--{split}_eval_results", "")
        with open(f) as fh:
            lookup[model] = json.load(fh)
    return lookup

### 4d. Match samples to labels and build the records

For each sample file in `complete/` and `instruct/`, we:
1. Extract the model identifier from the filename
2. Try to find the corresponding eval labels (manual map first, then fuzzy normalization)
3. If we find a match, read every line in the sample file and pair each solution with its pass/fail label
4. If we can't match, skip the file entirely (no labels means no training signal)

Note that the Llama 3.1 variants (8B and 70B) are mapped to the same eval names as Llama 3.0. This is because BigCodeBench published evaluation results under the Llama 3 name but the zip includes both 3.0 and 3.1 sample files. The tasks and evaluation criteria are identical, so the labels apply to both.

In [14]:
# load the task descriptions so we can merge metadata later
tasks = {}
with open(TASKS_PATH) as fh:
    for line in fh:
        t = json.loads(line)
        tasks[t["task_id"]] = t

print(f"Loaded {len(tasks)} task descriptions.")

Loaded 1140 task descriptions.


In [15]:
records = []

for split in ["complete", "instruct"]:
    sample_subdir = SAMPLES_DIR / split
    if not sample_subdir.exists():
        print(f"Warning: {sample_subdir} not found, skipping.")
        continue

    # load eval results and build a normalized lookup for fuzzy matching
    eval_lookup = load_eval_lookup(split)
    eval_normalized = {normalize_model_name(k): (k, v) for k, v in eval_lookup.items()}

    matched_count = 0
    skipped_models = []

    sample_files = sorted(sample_subdir.glob("*.jsonl"))
    for sample_file in tqdm(sample_files, desc=f"Processing {split}"):
        # the filename format is "model_id--bigcodebench-*.jsonl"
        # we split on "--bigcodebench" to get the model identifier
        sample_model = sample_file.stem.split("--bigcodebench")[0]

        # try the manual map first, then fall back to fuzzy normalization
        if sample_model in MANUAL_MAP:
            eval_model_name = MANUAL_MAP[sample_model]
            labels = eval_lookup.get(eval_model_name)
        else:
            match = eval_normalized.get(normalize_model_name(sample_model))
            if match:
                eval_model_name, labels = match
            else:
                eval_model_name, labels = None, None

        if labels is None:
            skipped_models.append(sample_model)
            continue

        matched_count += 1
        with open(sample_file) as fh:
            for line in fh:
                row = json.loads(line)
                task_id = row["task_id"]
                label = labels.get(task_id)
                if label is None:
                    continue
                records.append({
                    "task_id":    task_id,
                    "model_name": eval_model_name,
                    "split":      split,
                    "solution":   row.get("solution", ""),
                    "label":      label,
                })

    print(f"{split}: matched {matched_count} models, skipped {len(skipped_models)} (no eval results)")
    if skipped_models:
        print(f"  Skipped models: {skipped_models[:10]}{'...' if len(skipped_models) > 10 else ''}")

print(f"\nTotal records before merging task metadata: {len(records):,}")

Processing complete:   0%|          | 0/87 [00:00<?, ?it/s]

complete: matched 56 models, skipped 31 (no eval results)
  Skipped models: ['01-ai--Yi-Coder-9B-Chat', 'Artigenz--Artigenz-Coder-DS-6.7B', 'CohereForAI--c4ai-command-r-08-2024', 'NTQAI--Nxcode-CQ-7B-orpo', 'Nexusflow--Athene-70B', 'NousResearch--Hermes-2-Pro-Llama-3-70B', 'Phind--Phind-CodeLlama-34B-v2', 'Qwen--Qwen2.5-0.5B-Instruct', 'Qwen--Qwen2.5-1.5B-Instruct', 'Qwen--Qwen2.5-14B-Instruct']...


Processing instruct:   0%|          | 0/134 [00:00<?, ?it/s]

instruct: matched 54 models, skipped 80 (no eval results)
  Skipped models: ['01-ai--Yi-Coder-9B-Chat', '01-ai--Yi-Coder-9B-Chat', '01-ai--Yi-Coder-9B-Chat', 'Artigenz--Artigenz-Coder-DS-6.7B', 'CohereForAI--c4ai-command-r-08-2024', 'CohereForAI--c4ai-command-r-plus-08-2024', 'CohereForAI--c4ai-command-r-plus-08-2024', 'NTQAI--Nxcode-CQ-7B-orpo', 'Nexusflow--Athene-70B', 'NousResearch--Hermes-2-Pro-Llama-3-70B']...

Total records before merging task metadata: 123,416


### 4e. Merge task metadata and save

We merge the task-level information (prompts, required libraries, entry point) directly into the samples dataframe. This produces a single flat CSV where every row has both the generated code and the context it was generated from. This is more convenient for ML workflows than having to join separate tables.

The `complete` and `instruct` splits are kept as separate rows because the same model produces different code depending on the prompt format, and pass rates differ meaningfully between the two (roughly 45% for complete vs. 37% for instruct).

In [16]:
samples_df = pd.DataFrame(records)

tasks_df = pd.DataFrame([
    {
        "task_id":         t["task_id"],
        "complete_prompt": t.get("complete_prompt", ""),
        "instruct_prompt": t.get("instruct_prompt", ""),
        "libs":            t.get("libs", ""),
        "entry_point":     t.get("entry_point", ""),
    }
    for t in tasks.values()
])

samples_df = samples_df.merge(tasks_df, on="task_id", how="left")

print(f"Final dataset: {len(samples_df):,} rows, {samples_df['model_name'].nunique()} models")
print(f"Columns: {list(samples_df.columns)}")

Final dataset: 123,416 rows, 57 models
Columns: ['task_id', 'model_name', 'split', 'solution', 'label', 'complete_prompt', 'instruct_prompt', 'libs', 'entry_point']


In [17]:
output_path = PROCESSED / "samples.csv"
samples_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Saved to processed/samples.csv


## Step 5: Verify the output

Let's do a few sanity checks to make sure the dataset looks right.

In [18]:
df = pd.read_csv(output_path)
print(f"Shape: {df.shape}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())
df.head(3)

Shape: (123416, 9)

Column types:
task_id            object
model_name         object
split              object
solution           object
label               int64
complete_prompt    object
instruct_prompt    object
libs               object
entry_point        object
dtype: object

Missing values:
task_id             0
model_name          0
split               0
solution           72
label               0
complete_prompt     0
instruct_prompt     0
libs                0
entry_point         0
dtype: int64


,task_id,model_name,split,solution,label,complete_prompt,instruct_prompt,libs,entry_point
0,BigCodeBench/0,Yi_1.5_34B_Chat,complete,import itertools\nfrom random import shuffle\n...,0,import itertools\nfrom random import shuffle\n...,Calculates the average of the sums of absolute...,"['random', 'itertools']",task_func
1,BigCodeBench/1,Yi_1.5_34B_Chat,complete,import collections\nimport random\nimport stri...,1,import collections\nimport random\nimport stri...,Generate a random string of the specified leng...,"['collections', 'random', 'string']",task_func
2,BigCodeBench/2,Yi_1.5_34B_Chat,complete,import random\nimport statistics\nimport rando...,1,import random\nimport statistics\n\ndef task_f...,Create a dictionary in which keys are random l...,"['statistics', 'random']",task_func


In [19]:
print(f"Overall pass rate: {df['label'].mean():.1%}")
print(f"\nPass rate by split:")
for split, group in df.groupby("split"):
    print(f"  {split}: {len(group):,} samples, {group['label'].mean():.1%} pass rate")

print(f"\nUnique models: {df['model_name'].nunique()}")
print(f"Unique tasks: {df['task_id'].nunique()}")

Overall pass rate: 41.2%

Pass rate by split:
  complete: 63,840 samples, 45.5% pass rate
  instruct: 59,576 samples, 36.5% pass rate

Unique models: 57
Unique tasks: 1140


In [20]:
# pass rate by model, top 10 and bottom 10
model_pass_rates = df.groupby("model_name")["label"].mean().sort_values(ascending=False)

print("Top 10 models by pass rate:")
for name, rate in model_pass_rates.head(10).items():
    print(f"  {name:45s} {rate:.1%}")

print(f"\nBottom 10 models by pass rate:")
for name, rate in model_pass_rates.tail(10).items():
    print(f"  {name:45s} {rate:.1%}")

Top 10 models by pass rate:
  DeepSeek_Coder_V2_Instruct                    54.0%
  GPT_4_Turbo_2024_04_09                        53.2%
  ReflectionCoder_DS_33B                        53.0%
  Codestral_22B_v0.1                            52.8%
  GPT_4o_2024_05_13                             52.5%
  GPT_4_0613                                    51.6%
  Claude_3_Opus_20240229                        51.4%
  Hermes_2_Theta_Llama_3_70B                    50.6%
  Gemini_1.5_Pro_API_0514                       50.6%
  Gemini_1.5_Flash_API_0514                     49.3%

Bottom 10 models by pass rate:
  Llama_3_8B_Instruct                           34.4%
  Mistral_Large_2402                            34.2%
  CodeLlama_34B_Instruct                        32.3%
  Granite_Code_3B_Instruct                      31.4%
  CodeLlama_13B_Instruct                        30.1%
  Yi_1.5_6B_Chat                                29.7%
  DeepSeek_Coder_1.3B_Instruct                  26.2%
  OpenCodeInterpreter_

In [21]:
# check solution length distribution
df["solution_len"] = df["solution"].str.len()
print(f"Solution length (characters):")
print(df["solution_len"].describe().to_string())

Solution length (characters):
count    123344.000000
mean       1146.913689
std         590.450736
min           9.000000
25%         707.000000
50%        1049.000000
75%        1478.000000
max        6861.000000


The dataset is ready. It contains 123,416 labeled code samples from 57 LLMs across 1,140 programming tasks, each with two prompt variants. Every row has the generated code, a binary pass/fail label, and the task context (prompts, required libraries, entry point).

In [23]:
## 6. Pipeline Summary <a id="6-pipeline-summary"></a>

```
                          DATA PIPELINE FLOW

    Source 1                Source 2                 Source 3
    HuggingFace            GitHub Releases          HuggingFace
    bigcode/bigcodebench   bigcodebench v0.2.5      bigcodebench-*-perf
         |                      |                        |
         v                      v                        v
    1,140 tasks            158 MB zip               137 JSON files
    (prompts, tests,       (100+ LLMs x             (per-model
     entry points)          1,140 tasks)              pass/fail)
         |                      |                        |
         |               -------+-------                 |
         |               |             |                 |
         |               v             v                 |
         |          complete/      instruct/             |
         |          87 models      134 models            |
         |               |             |                 |
         |               +------+------+                 |
         |                      |                        |
         |                      v                        |
         |              Name Normalization                |
         |              (fuzzy + manual map)              |
         |                      |                        |
         |                      +<-----------------------+
         |                      |
         |                      v
         |              Match & Join
         |              57 models matched
         |              (others dropped: no labels)
         |                      |
         +--------------------->+
                                |
                                v
                        Merge Task Metadata
                        (prompts, libs, entry_point)
                                |
                                v
                       samples.csv
                       123,416 rows x 9 columns
                       57 models, 1,140 tasks
                       41% overall pass rate
```